# GVH–CHH — Jupiter H0 : réplication temporelle sur octobre 2026

**Notebook :** `GVH_CHH_Jupiter_Horizons_H0_Independent_Oct2026_Temporal_Replication_Gate_FAST`  
**Version :** draft v0.2, 20 novembre 2026 ; protocole CHH explicitement réaffirmé (non exécuté sur les nouvelles éphémérides dans cette livraison).  
**Objectif :** reproduire l'étude H0 d'ablation de septembre sur une période disjointe, **sans réoptimiser le protocole à la lumière des résultats de septembre**.  
**Point de départ archivé :** `archive/gvh-chh-2026-11-20/archives/gvh-chh/2026-11-20/jupiter-h0-final/`.  
**Architecture à quatre niveaux :** (1) CHH : hypothèse / séparation conceptuelle ; (2) mécanique gravitationnelle établie ; (3) tests chiffrés et réplicables ; (4) contrôles et verdict strictement limité.

**Catégories :** `CHH-STRUCTURE`, `KNOWN-PHYSICS`, `NUMERICAL-TEST`, `OPEN-LOCK` (et `AUX-DERIVATIVE` seulement si une dérivée collective auxiliaire est effectivement choisie). Aucun ajout de force GVH, aucune lune cachée, aucun résultat astrométrique indépendant allégué.

## Protocole figé — CHH sans dérivée globale intrinsèque

**Référence obligatoire :** `GVH_CHH_PROTOCOL_FROZEN_v1.0` (18 novembre 2026). Ce notebook applique la règle méthodologique sans la modifier.

- **`CHH-STRUCTURE` :** la Composition hétérodynamique hiérarchique est une **composition de sous-cubes hétérogènes**. En tant qu'opération de composition, elle **n'implique aucune dérivée globale intrinsèque** et n'impose aucune équation différentielle universelle :
  
  $C_{S,\ell}=\mathcal A_{\ell,\tau}(\{C_i,D_i\},\{I_{ij}\};\boldsymbol\theta)$.

  L'agrégation dépend de la grandeur physique $\tau$, de ses unités et du modèle choisi. La composition ne doit pas être assimilée à une dérivation de loi dynamique.

- **`AUX-DERIVATIVE` (facultatif) :** pour faciliter le calcul d'une grandeur collective explicite $Y_S(t)$, il est permis de **choisir** une projection ou un modèle effectif et d'en calculer $\dot Y_S$ ou $\ddot Y_S$. Cette dérivée est **auxiliaire et conditionnelle** : déclarer $Y_S$, ses unités, la composition pour $\tau$, les hypothèses, la loi externe invoquée, les erreurs et le domaine de validité. Elle ne devient jamais une loi inhérente à CHH.

- **`KNOWN-PHYSICS` :** les accélérations newtoniennes et les gradients du potentiel zonal de Jupiter ($J_2,J_4,J_6$), employés ici pour l'intégration et l'optimisation numérique, proviennent de **la mécanique gravitationnelle établie**. Ni leurs dérivées temporelles ni leurs gradients spatiaux ne sont postulés par CHH.

- **`EOM-GVH` :** une équation dérivée, le cas échéant, par variation d'une action $S_{\rm cand}$ appartient exclusivement à cette branche, **pas à la CHH**. Aucune EOM-GVH n'est testée dans le présent notebook.

**Séparation des quatre niveaux :** (1) CHH/composition ; (2) physique connue ; (3) expériences numériques ; (4) verdict et questions ouvertes. Le PASS technique n'est ni une validation empirique indépendante ni une preuve d'originalité physique.


## Niveau 1 — Plan préenregistré, avant lecture des nouvelles données

- **Période nouvelle fixée :** 1er octobre 2026 à 00 h TDB → 25 octobre 2026 à 00 h TDB, échantillonnage toutes les 6 heures (97 époques). Elle ne chevauche pas la période précédente du 1er au 25 septembre 2026.
- **Même centre et repère :** Jupiter (`500@599`), ICRF, vecteurs géométriques, km et km/s, pas de correction de propagation de la lumière.
- **Même forces :** monopôle, J2, interactions newtoniennes mutuelles, forçage solaire prescrit, J4/J6. Six ablations fixes.
- **Cibles évaluées :** Io et Ganymède. Europe et Callisto servent au modèle à quatre lunes.
- **Entraînement :** jours 0–10 et 0–15 ; réserve commune : jours strictement **postérieurs au 18e jour**. Douze corrections de position/vitesse initiale, et uniquement celles d'Io et Ganymède.
- **Critère principal de réplication :** `J2_MUTUAL` contre `KEPLER`, qui n'utilisent aucun vecteur du Soleil sur la réserve. Les variantes avec Soleil sont secondaires, étiquetées avec forçage externe connu.
- **Exclusion :** aucune conclusion de prédiction observationnelle indépendante ; Horizons est une éphéméride calculée à partir d'observations et de modèles physiques.

**Important :** le modèle solaire lit les vecteurs JPL du Soleil jusque dans les dates réservées. Les évaluations qui l'utilisent sont dites *à forçage externe prescrit*, pas « aveugles » vis-à-vis de toutes les entrées.

In [1]:
from pathlib import Path
import json, hashlib, io, csv, urllib.parse, urllib.request, shutil
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicHermiteSpline
from scipy.optimize import least_squares

NAME = 'GVH_CHH_Jupiter_Horizons_H0_Independent_Oct2026_Temporal_Replication_Gate_FAST'
ROOT = Path('/content') if Path('/content').exists() else Path('/mnt/data')
SOURCE = ROOT/'GVH_CHH_JUPITER_H0_OCT2026_INPUTS'
OUT = ROOT/'GVH_CHH_JUPITER_H0_OCT2026_REPLICATION'
SOURCE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
START, STOP, STEP = '2026-10-01', '2026-10-25', '6 h'
MOONS=('Io','Europa','Ganymede','Callisto')
TARGETS=('Io','Ganymede')
ID={'Io':501,'Europa':502,'Ganymede':503,'Callisto':504,'Sun':10}
TRAIN_DAYS=(10.0,15.0)
COMMON_RESERVED_START_D=18.0
MODELS=('KEPLER','J2_ONLY','MUTUAL_ONLY','J2_MUTUAL','J2_MUTUAL_SUN','J2_J4_J6_MUTUAL_SUN')
MAX_NFEV=14
GM_J=126686534.0
R_J=71492.0
GM_MOONS=np.array([5959.916,3202.739,9887.834,7179.29],float)
GM_SUN=132712440041.93938
J2=0.0146965735;J4=-0.0005866085;J6=0.0000342007
RA,DEC=np.deg2rad([268.056595,64.495303])
POLE=np.array([np.cos(DEC)*np.cos(RA),np.cos(DEC)*np.sin(RA),np.sin(DEC)])

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
print('Preregistered period:',START,'to',STOP,'| models:',len(MODELS))

Preregistered period: 2026-10-01 to 2026-10-25 | models: 6


## Niveau 2 — Acquisition JPL Horizons et conservation de la réponse brute

Aucune simulation synthétique ne remplace les données si l'API échoue. Les cinq réponses originales au format JSON sont **conservées en bytes tels que reçus** dans l'archive finale ; un CSV de vecteurs est également généré pour chacune. Toutes les réponses doivent avoir 97 dates identiques à pas de six heures, sous le centre demandé. Le notebook doit avoir accès au réseau.

In [2]:
def fetch_vectors(ident,label):
    q={'format':'json','COMMAND':f"'{ident}'",'OBJ_DATA':"'NO'",'MAKE_EPHEM':"'YES'",
       'EPHEM_TYPE':"'VECTORS'",'CENTER':"'500@599'",'REF_PLANE':"'FRAME'",'REF_SYSTEM':"'ICRF'",
       'START_TIME':f"'{START}'",'STOP_TIME':f"'{STOP}'",'STEP_SIZE':f"'{STEP}'",
       'OUT_UNITS':"'KM-S'",'VEC_TABLE':"'2'",'VEC_CORR':"'NONE'",'CSV_FORMAT':"'YES'"}
    url='https://ssd.jpl.nasa.gov/api/horizons.api?'+urllib.parse.urlencode(q)
    req=urllib.request.Request(url,headers={'User-Agent':'research-GVH-CHH/1.0'})
    with urllib.request.urlopen(req,timeout=90) as response:
        body=response.read()
    if len(body)>12_000_000:
        raise RuntimeError('Unexpectedly large JPL reply')
    result=json.loads(body.decode('utf-8')).get('result','')
    if '$$SOE' not in result or '$$EOE' not in result:
        raise RuntimeError(f'Incomplete Horizons reply for {label}: {result[:200]}')
    raw_path=SOURCE/f'HORIZONS_{label}_API_RAW.json'
    raw_path.write_bytes(body)
    span=result.split('$$SOE',1)[1].split('$$EOE',1)[0]
    records=[]
    for row in csv.reader(io.StringIO(span)):
        if len(row)<8:continue
        try: records.append([float(row[0])]+[float(x) for x in row[2:8]])
        except (ValueError,TypeError): pass
    arr=np.asarray(records,float)
    if arr.shape!=(97,7) or not np.all(np.isfinite(arr)):
        raise ValueError(f'Bad Horizons shape/values for {label}: {arr.shape}')
    if not np.allclose(np.diff(arr[:,0]),0.25,atol=2e-7,rtol=0):
        raise ValueError(f'Incorrect cadence for {label}')
    csv_path=SOURCE/f'HORIZONS_{label}_ICRF.csv'
    csv_path.write_text('JD_TDB,x_km,y_km,z_km,vx_km_s,vy_km_s,vz_km_s\n'+
                        '\n'.join(','.join(format(x,'.16g') for x in row) for row in arr)+'\n')
    return dict(name=label,naif_id=ident,samples=len(arr),request_url=url,
                raw_response_sha256=sha256(raw_path),csv_sha256=sha256(csv_path),
                raw_response_filename=raw_path.name,csv_filename=csv_path.name)

provenance=[]
for label,ident in ID.items():
    record=fetch_vectors(ident,label)
    provenance.append(record)
    print('JPL downloaded',label,record['samples'],'epochs')
provenance_path=SOURCE/'JPL_REQUEST_PROVENANCE.json'
provenance_path.write_text(json.dumps({
    'acquisition':'JPL_HORIZONS_API_REAL_NO_FALLBACK',
    'start':START,'stop':STOP,'step':STEP,'center':'500@599',
    'ref_system':'ICRF','out_units':'KM-S','vec_corr':'NONE',
    'responses':provenance
},ensure_ascii=False,indent=2))
print('Saved provenance:',provenance_path)

JPL downloaded Io 97 epochs
JPL downloaded Europa 97 epochs
JPL downloaded Ganymede 97 epochs
JPL downloaded Callisto 97 epochs
JPL downloaded Sun 97 epochs
Saved provenance: /content/GVH_CHH_JUPITER_H0_OCT2026_INPUTS/JPL_REQUEST_PROVENANCE.json


In [3]:
need=['JD_TDB','x_km','y_km','z_km','vx_km_s','vy_km_s','vz_km_s']
inputs={}
for k in ID:
    f=SOURCE/f'HORIZONS_{k}_ICRF.csv'
    df=pd.read_csv(f)
    if list(df.columns)!=need:raise ValueError('Unexpected CSV columns for '+k)
    a=df.to_numpy(dtype=float)
    if a.shape!=(97,7) or not np.all(np.isfinite(a)):raise ValueError('Invalid CSV '+k)
    inputs[k]=a
jd=inputs['Io'][:,0]
for k,a in inputs.items():
    if not np.allclose(a[:,0],jd,atol=1e-8,rtol=0):raise ValueError('Non-aligned '+k)
if not np.allclose(np.diff(jd),0.25,atol=2e-7,rtol=0):raise ValueError('Invalid grid cadence')
start_jd=float(pd.Timestamp(START).to_julian_date())
stop_jd=float(pd.Timestamp(STOP).to_julian_date())
if abs(jd[0]-start_jd)>0.01 or abs(jd[-1]-stop_jd)>0.01:
    raise ValueError('Wrong period; possible stale or mixed CSV inputs')
# September 1-25 ends a month before this fixed replication period.
assert jd[0]>float(pd.Timestamp('2026-09-25').to_julian_date())
source_hashes={p.name:sha256(p) for p in SOURCE.iterdir() if p.is_file()}
epochs=(jd-jd[0])*86400.0
obs=np.stack([inputs[k][:,1:4] for k in MOONS],axis=1)
x0=np.stack([inputs[k][0,1:7] for k in MOONS],axis=0)
sun=inputs['Sun']
sun_spline=CubicHermiteSpline(epochs,sun[:,1:4],sun[:,4:7],axis=0)
assert max(TRAIN_DAYS)<COMMON_RESERVED_START_D<epochs[-1]/86400
print('REAL OCTOBER JPL REFERENCE VERIFIED: 97 dates, 5 CSVs, 5 raw API replies')

REAL OCTOBER JPL REFERENCE VERIFIED: 97 dates, 5 CSVs, 5 raw API replies


## Niveau 3 — Physique établie, tests d'intégrateur, ajustements sans fuite de cibles

Le modèle est repris **sans changer les constantes ni la formule des forces** du notebook H0 de septembre. On ajuste uniquement les douze corrections aux positions et vitesses initiales des deux lunes cibles, sur les dates d'entraînement. Les états initiaux d'Europe et Callisto restent fixés. Les modèles utilisant le Soleil disposent de son éphéméride exogène sur toute la période (test **conditionnel**, non totalement aveugle).

In [4]:
def legendre_even(n,s):
    if n==2: return (3*s*s-1)/2, 3*s
    if n==4: return (35*s**4-30*s*s+3)/8,(140*s**3-60*s)/8
    if n==6: return (231*s**6-315*s**4+105*s*s-5)/16,(1386*s**5-1260*s**3+210*s)/16
    raise ValueError(n)

def zonal_potential(r, orders=(2,4,6)):
    rr=np.linalg.norm(r); s=np.dot(POLE,r)/rr
    J={2:J2,4:J4,6:J6}
    return -GM_J/rr*(1-sum(J[n]*(R_J/rr)**n*legendre_even(n,s)[0] for n in orders))

def zonal_accel(r,orders=(2,4,6)):
    rr=np.linalg.norm(r); rh=r/rr; s=np.dot(POLE,rh)
    J={2:J2,4:J4,6:J6}; a=np.zeros(3)
    for n in orders:
        P,dP=legendre_even(n,s)
        a+=GM_J*J[n]*R_J**n/rr**(n+2)*((n+1)*P*rh-dP*(POLE-s*rh))
    return a

def grad_check():
    x=np.array([6.2e5,-3e5,4.4e4]); h=8.0; eye=np.eye(3)
    grad=np.array([(zonal_potential(x+h*eye[k])-zonal_potential(x-h*eye[k]))/(2*h) for k in range(3)])
    expected=-GM_J*x/np.linalg.norm(x)**3+zonal_accel(x)
    return np.linalg.norm(-grad-expected)/np.linalg.norm(expected)
assert grad_check() < 3e-6

ABLATION={
 'KEPLER':((),False,False),
 'J2_ONLY':((2,),False,False),
 'MUTUAL_ONLY':((),True,False),
 'J2_MUTUAL':((2,),True,False),
 'J2_MUTUAL_SUN':((2,),True,True),
 'J2_J4_J6_MUTUAL':((2,4,6),True,False),
 'J2_J4_J6_MUTUAL_SUN':((2,4,6),True,True)
}

def rhs(t,y,variant):
    orders,mutual,solar=ABLATION[variant]
    st=y.reshape(4,6); pos=st[:,:3]; vel=st[:,3:]; acc=np.zeros((4,3))
    for i in range(4):
        r=pos[i];rad=np.linalg.norm(r)
        acc[i]=-GM_J*r/rad**3+zonal_accel(r,orders)
        if mutual:
            for j in range(4):
                if i==j:continue
                d=pos[j]-r
                acc[i]+=GM_MOONS[j]*(d/np.linalg.norm(d)**3-pos[j]/np.linalg.norm(pos[j])**3)
        if solar:
            rs=sun_spline(t); d=rs-r
            acc[i]+=GM_SUN*(d/np.linalg.norm(d)**3-rs/np.linalg.norm(rs)**3)
    return np.column_stack((vel,acc)).reshape(-1)

def propagate(states,tt,variant):
    tt=np.asarray(tt,float)
    if abs(tt[0])>1e-8 or not np.all(np.diff(tt)>0):raise ValueError('invalid times')
    sol=solve_ivp(lambda t,y:rhs(t,y,variant),(0,float(tt[-1])),states.ravel(),
                  method='DOP853',t_eval=tt,rtol=3e-11,atol=4e-10,max_step=21600.)
    if not sol.success:raise RuntimeError(sol.message)
    return sol.y.T.reshape(len(tt),4,6)

# Offline integrator smoke: circular orbit remains approximately circular under Kepler.
synthetic=np.zeros((4,6))
for k,r in enumerate((420000.,670000.,1070000.,1880000.)):
    synthetic[k,:3]=[r,0,0];synthetic[k,3:]=[0,np.sqrt(GM_J/r),0]
test=propagate(synthetic,np.linspace(0,2*86400,9),'KEPLER')
radial_error=np.max(np.abs(np.linalg.norm(test[:,0,:3],axis=1)-420000.))
assert radial_error<1e-3
print('PASS force/integrator unit checks;',grad_check(),radial_error)

PASS force/integrator unit checks; 8.339531982418163e-11 2.7468428015708923e-05


In [5]:
days=epochs/86400.0
COMMON=np.flatnonzero(days>COMMON_RESERVED_START_D)
TARGET_IDX=[0,2]
REPORT={
    'notebook':NAME,'version':'draft_v0.1_oct2026_real_run',
    'state':'OCT2026_JPL_H0_REPLICATION_RUNNING',
    'source_period':{'start':START,'stop':STOP,'cadence':'6h'},
    'source_hashes_sha256':source_hashes,
    'source_api_provenance':provenance,
    'h0_hidden_body_count':0,'GVH_USED':False,'EOM_GVH':False,'ALMA_USED':False,
    'target_moons':list(TARGETS),'train_windows_days':list(TRAIN_DAYS),
    'common_reserved_start_day_exclusive':COMMON_RESERVED_START_D,
    'model_names':list(MODELS),'nuisance_parameter_count':12,
    'external_sun_forcing_uses_reserved_ephemerides':True,
    'primary_comparison_without_solar_external_forcing':['KEPLER','J2_MUTUAL'],
    'holdout_is_JPL_reference_ephemeris_not_independent_raw_astrometry':True,
    'nonlinear_optimizer_nfev_budget':MAX_NFEV,'results':[]
}

def initial_shift(z):
    a=x0.copy();shift=np.asarray(z).reshape(2,6)
    for j,i in enumerate(TARGET_IDX):
        a[i,:3]+=1000*shift[j,:3]
        a[i,3:]+=0.01*shift[j,3:]
    return a

def fit_resid(z,ix,variant):
    pred=propagate(initial_shift(z),epochs[ix],variant)
    return ((pred[:,TARGET_IDX,:3]-obs[ix][:,TARGET_IDX,:])/1000).reshape(-1)

rows=[]
for cutoff in TRAIN_DAYS:
    train=np.flatnonzero(days<=cutoff+1e-9)
    assert max(train)<min(COMMON)
    nearby=np.flatnonzero((days>cutoff)&(days<=cutoff+3.0))
    print('Training:',cutoff,'days | train dates:',len(train),'| reserved:',len(COMMON))
    for variant in MODELS:
        fit=least_squares(fit_resid,np.zeros(12),args=(train,variant),method='trf',
                          max_nfev=MAX_NFEV,xtol=1e-6,ftol=1e-6,gtol=1e-6)
        pred=propagate(initial_shift(fit.x),epochs,variant)
        res=pred[:,TARGET_IDX,:3]-obs[:,TARGET_IDX,:]
        rmse=lambda ix,values: float(np.sqrt(np.mean(values[ix]**2)))
        entry={
          'window_train_days':float(cutoff),'model':variant,
          'optimizer_success':bool(fit.success),'optimizer_message':str(fit.message),
          'optimizer_nfev':int(fit.nfev),'train_RMSE_xyz_km':rmse(train,res),
          'common_reserved_RMSE_xyz_km':rmse(COMMON,res),
          'nearby_3d_RMSE_xyz_km':rmse(nearby,res) if len(nearby) else None,
          'Io_reserved_RMSE_xyz_km':rmse(COMMON,res[:,0,:]),
          'Ganymede_reserved_RMSE_xyz_km':rmse(COMMON,res[:,1,:]),
          'initial_state_corrections_scaled':fit.x.tolist(),
          'training_samples':int(len(train)),'reserved_samples':int(len(COMMON))
        }
        REPORT['results'].append(entry)
        rows.append({k:v for k,v in entry.items() if k!='initial_state_corrections_scaled'})
        print(variant, 'reserved RMSE:',round(entry['common_reserved_RMSE_xyz_km'],5),
              'fit:',fit.success)
metrics=pd.DataFrame(rows)
metrics.to_csv(OUT/(NAME+'_metrics.csv'),index=False)
REPORT['all_optimizer_runs_converged']=bool(all(e['optimizer_success'] for e in REPORT['results']))
REPORT['state']='OCT2026_JPL_H0_REPLICATION_EXECUTED' if REPORT['all_optimizer_runs_converged'] else 'INCOMPLETE_NONCONVERGED_OPTIMIZERS'
print(metrics[['window_train_days','model','common_reserved_RMSE_xyz_km','optimizer_success']].to_string(index=False))

Training: 10.0 days | train dates: 41 | reserved: 24
KEPLER reserved RMSE: 380.92592 fit: True
J2_ONLY reserved RMSE: 398.15232 fit: True
MUTUAL_ONLY reserved RMSE: 70.3788 fit: True
J2_MUTUAL reserved RMSE: 16.88092 fit: True
J2_MUTUAL_SUN reserved RMSE: 18.35094 fit: True
J2_J4_J6_MUTUAL_SUN reserved RMSE: 18.33909 fit: True
Training: 15.0 days | train dates: 61 | reserved: 24
KEPLER reserved RMSE: 283.31474 fit: True
J2_ONLY reserved RMSE: 308.28202 fit: True
MUTUAL_ONLY reserved RMSE: 60.69191 fit: True
J2_MUTUAL reserved RMSE: 12.72717 fit: True
J2_MUTUAL_SUN reserved RMSE: 13.19678 fit: True
J2_J4_J6_MUTUAL_SUN reserved RMSE: 13.18234 fit: True
 window_train_days               model  common_reserved_RMSE_xyz_km  optimizer_success
              10.0              KEPLER                   380.925921               True
              10.0             J2_ONLY                   398.152325               True
              10.0         MUTUAL_ONLY                    70.378803             

In [6]:
contrast_pairs=(
 ('J2 contribution without mutual','KEPLER','J2_ONLY'),
 ('Mutual contribution without J2','KEPLER','MUTUAL_ONLY'),
 ('Mutual contribution with J2','J2_ONLY','J2_MUTUAL'),
 ('J2 contribution with mutual','MUTUAL_ONLY','J2_MUTUAL'),
 ('Sun given J2 and mutual','J2_MUTUAL','J2_MUTUAL_SUN'),
 ('J4/J6 given J2 and mutual and Sun','J2_MUTUAL_SUN','J2_J4_J6_MUTUAL_SUN')
)
contrasts=[]
for w in TRAIN_DAYS:
    selected={r['model']:r for r in REPORT['results'] if r['window_train_days']==w}
    for name,base,enh in contrast_pairs:
        a=selected[base]['common_reserved_RMSE_xyz_km']
        b=selected[enh]['common_reserved_RMSE_xyz_km']
        contrasts.append(dict(train_days=w,contrast=name,base=base,enhanced=enh,
                              base_rmse_km=a,enhanced_rmse_km=b,reduction_percent=100*(a-b)/a))
contrast_df=pd.DataFrame(contrasts)
contrast_df.to_csv(OUT/(NAME+'_contrasts.csv'),index=False)
print('Contrasts saved (including primary KEPLER vs J2_MUTUAL diagnostic)')

Contrasts saved (including primary KEPLER vs J2_MUTUAL diagnostic)


## Niveau 4 — Audit technique et verdict limité

Un PASS signifie que **l'expérience d'octobre a été exécutée correctement** selon ses vérifications internes. Il n'implique ni validation externe de la théorie GVH, ni succès de réplication au sens d'un seuil d'erreur présélectionné, ni données astrométriques indépendantes. Le tableau de septembre est un comparateur descriptif historique, non une contrainte d'ajustement de l'expérience d'octobre.

In [7]:
checks={
 'five_real_jpl_csv_exist': all((SOURCE/f'HORIZONS_{k}_ICRF.csv').is_file() for k in ID),
 'five_unmodified_api_replies_saved': all((SOURCE/f'HORIZONS_{k}_API_RAW.json').is_file() for k in ID),
 'source_hashes_present':len(source_hashes)==11 and all(source_hashes.values()),
 'matching_time_grid':all(np.allclose(inputs[k][:,0],jd,atol=1e-8,rtol=0) for k in ID),
 'october_period_disjoint_from_september':abs(jd[0]-start_jd)<0.01 and abs(jd[-1]-stop_jd)<0.01,
 'reserved_after_training':all(max(np.flatnonzero(days<=w))<min(COMMON) for w in TRAIN_DAYS),
 'fixed_12_nuisance_states':all(len(e['initial_state_corrections_scaled'])==12 for e in REPORT['results']),
 'no_hidden_bodies':REPORT['h0_hidden_body_count']==0,
 'no_gvh_or_alma':not REPORT['GVH_USED'] and not REPORT['EOM_GVH'] and not REPORT['ALMA_USED'],
 'zonal_gradient_math':bool(grad_check()<3e-6),
 'kepler_propagation_smoke':bool(radial_error<1e-3),
 'fit_convergence':REPORT['all_optimizer_runs_converged'],
 'expected_rows':len(REPORT['results'])==len(MODELS)*len(TRAIN_DAYS)
}
REPORT['checks']=checks
REPORT['scientific_verdict']=(
  'PASS_TECHNICAL_OCT2026_TEMPORAL_REPLICATION__JPL_REFERENCE_ONLY__CHH_UNTESTED'
  if all(checks.values()) else 'FAIL_OR_INCOMPLETE_OCT2026_H0__NO_CHH_CLAIM'
)
REPORT['contrasts']=contrasts
REPORT['residuals_are_not_observation_noise']=True
result=OUT/(NAME+'_results.json')
result.write_text(json.dumps(REPORT,ensure_ascii=False,indent=2,default=float))
(OUT/'SHA256_INPUTS.txt').write_text(''.join(f'{h}  {name}\n' for name,h in sorted(source_hashes.items())))
print('Checks:',sum(checks.values()),'/',len(checks))
print('Verdict:',REPORT['scientific_verdict'])
print('Results:',result)

Checks: 13 / 13
Verdict: PASS_TECHNICAL_OCT2026_TEMPORAL_REPLICATION__JPL_REFERENCE_ONLY__CHH_UNTESTED
Results: /content/GVH_CHH_JUPITER_H0_OCT2026_REPLICATION/GVH_CHH_Jupiter_Horizons_H0_Independent_Oct2026_Temporal_Replication_Gate_FAST_results.json


In [8]:
# Packaging only after successful execution and preservation of evidence.
assert result.is_file(), 'Run the audit cell before packaging.'
assert all(checks.values()), 'Technical checks not passed: inspect report; do not mark as validated.'
archive=ROOT/'GVH_CHH_JUPITER_OCT2026_REPLICATION_RESULTS.zip'
with __import__('zipfile').ZipFile(archive,'w',compression=__import__('zipfile').ZIP_DEFLATED) as z:
    for directory in (SOURCE, OUT):
        for p in sorted(directory.iterdir()):
            if p.is_file(): z.write(p,arcname=str(Path(directory.name)/p.name))
print('Final archive:',archive,'bytes:',archive.stat().st_size)
print('SHA-256:',sha256(archive))

Final archive: /content/GVH_CHH_JUPITER_OCT2026_REPLICATION_RESULTS.zip bytes: 86185
SHA-256: 4957c18459e95e1a609209fa882a4960adfa4e21c567148a0b2bda1a1f4ab2f0


### Sauvegarde facultative sur Drive (après l'audit)

La cellule ci-dessous est **désactivée par défaut**. Après succès du calcul et création du ZIP, passer `SAVE_TO_DRIVE=True`, l'exécuter et partager ensuite l'URL du fichier sauvegardé. Garder la nouvelle archive séparée des résultats H0 septembre déjà publiés sur GitHub.

In [ ]:
SAVE_TO_DRIVE=True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    folder=Path('/content/drive/MyDrive/GVH_CHH_Jupiter_Archives')
    folder.mkdir(parents=True,exist_ok=True)
    dest=folder/archive.name
    shutil.copy2(archive,dest)
    assert sha256(archive)==sha256(dest)
    print('Saved to Drive and SHA-256 verified:',dest)
else:
    print('Optional Drive save disabled. Set SAVE_TO_DRIVE=True after all checks pass.')